In [1]:
import numpy as np
import pandas as pd
from sklearn.decomposition import PCA


from sklearn.impute import SimpleImputer
from sklearn.model_selection import GroupKFold, cross_validate
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import accuracy_score, roc_auc_score


In [2]:
df_dataset = pd.read_parquet("../../data/processed/dataset_features.parquet")

df_dataset = df_dataset[df_dataset["condition"].isin(["EO", "EC"])]

df_dataset["y"] = df_dataset["condition"].map({"EO": 0, "EC": 1})

In [3]:
df_dataset.columns

Index(['degree_AF3', 'clustering_AF3', 'betweenness_AF3', 'hub_frequency_AF3',
       'sd_degree_AF3', 'bin_clustering_AF3', 'degree_AF4', 'clustering_AF4',
       'betweenness_AF4', 'hub_frequency_AF4',
       ...
       'age_65-70', 'age_70-75', 'age_75-80', 'gender_2', 'handedness_left',
       'handedness_right', 'education_gymnasium', 'education_realschule',
       'relationship_status_yes', 'y'],
      dtype='object', length=361)

In [4]:
NODE_FEATURES = [
    c for c in df_dataset.columns
    if c.startswith(("degree_", "clustering_", "betweenness_", "hub_"))
]

In [5]:
df_dataset = df_dataset.rename(columns={
    "Gender_ 1=female_2=male": "gender",
    "Age": "age",
    "Handedness": "handedness",
    "Education": "education",
    "DRUG_0=negative_1=Positive": "drug",
    "Smoking_num_(Non-smoker=1, Occasional Smoker=2, Smoker=3)": "smoking",
    "SKID_Diagnoses": "skid_diagnoses",
    "SKID_Diagnoses 1": "skid_diagnoses_1",
    "SKID_Diagnoses 2": "skid_diagnoses_2",
    "Comments_SKID_assessment": "comments_skid_assessment",
    "Hamilton_Scale": "hamilton_scale",
    "BSL23_sumscore": "bsl23_sumscore",
    "BSL23_behavior": "bsl23_behavior",
    "AUDIT": "audit",
    "Standard_Alcoholunits_Last_28days": "standard_alcoholunits_last_28days",
    "Alcohol_Dependence_In_1st-3rd_Degree_relative": "alcohol_dependence_in_1st_3rd_degree_relative",
    "Relationship_Status": "relationship_status"
})

In [6]:
META_FEATURES = ["age", "gender"]

In [7]:
def evaluate_model(X, y, model):
    # EO e EC do mesmo participante ficam sempre na mesma dobra
    cv = GroupKFold(n_splits=5)
    groups = df_dataset.loc[X.index, "subject_id"]

    scoring = {
        "acc": "accuracy",
        "auc": "roc_auc"
    }

    scores = cross_validate(
        model,
        X,
        y,
        cv=cv,
        groups=groups,
        scoring=scoring
    )

    acc_mean = scores["test_acc"].mean()
    acc_std = scores["test_acc"].std()

    auc_mean = scores["test_auc"].mean()
    auc_std = scores["test_auc"].std()

    print(
        "=== Model Evaluation (GroupKFold por participante, 5 folds) ===\n"
        f"Mean Accuracy : {acc_mean:.3f} ± {acc_std:.3f}\n"
        f"Mean ROC-AUC  : {auc_mean:.3f} ± {auc_std:.3f}\n"
        "Note: ROC-AUC values close to 0.5 indicate random performance."
    )

    return {
        "accuracy_mean": acc_mean,
        "auc_mean": auc_mean
    }


A função acima avalia um modelo de classificação da condição (EO vs EC) com validação cruzada agrupada por participante (GroupKFold, 5 folds): o EO e o EC de uma mesma pessoa ficam sempre na mesma dobra, para que o modelo não seja testado em alguém que já viu no treino. Calcula acurácia e ROC-AUC em cada fold e retorna as médias. Imputação (mediana), padronização e PCA ficam dentro do `Pipeline`, ajustados só com os dados de treino de cada dobra.

### Regressão Logística

In [8]:
X = df_dataset[NODE_FEATURES].select_dtypes(include="number")
y = df_dataset["y"]

pipe_lr = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
    ("clf", LogisticRegression(max_iter=1000))
])

evaluate_model(X, y, pipe_lr)


=== Model Evaluation (GroupKFold por participante, 5 folds) ===
Mean Accuracy : 0.639 ± 0.041
Mean ROC-AUC  : 0.626 ± 0.071
Note: ROC-AUC values close to 0.5 indicate random performance.


{'accuracy_mean': np.float64(0.639090909090909),
 'auc_mean': np.float64(0.6257851239669422)}

Com todas as métricas locais (212 variáveis), a regressão logística teve acurácia média de 63,9% ± 4,1% e ROC-AUC de 0,626 ± 0,071 na classificação EO vs EC: acima do acaso, mas com capacidade discriminativa modesta.

Numa versão anterior, com validação estratificada simples (EO e EC da mesma pessoa podiam cair em dobras diferentes), a AUC ficava abaixo de 0,5. Isso é um artefato típico de dados pareados: o modelo aprende a identidade do participante com o rótulo oposto. A validação agrupada por participante evita esse problema.

### Regressão Logística com PCA

In [9]:

pipe_pca_lr = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
    ("pca", PCA(n_components=10)),
    ("clf", LogisticRegression(max_iter=1000))
])

evaluate_model(X, y, pipe_pca_lr)


=== Model Evaluation (GroupKFold por participante, 5 folds) ===
Mean Accuracy : 0.605 ± 0.086
Mean ROC-AUC  : 0.547 ± 0.073
Note: ROC-AUC values close to 0.5 indicate random performance.


{'accuracy_mean': np.float64(0.6045454545454545),
 'auc_mean': np.float64(0.5472231404958677)}

Com PCA (10 componentes, ajustado dentro de cada dobra), a regressão logística teve acurácia de 60,5% ± 8,6% e ROC-AUC de 0,547 ± 0,073, mais perto do acaso que o modelo com todas as métricas. As primeiras componentes captam sobretudo a variação entre participantes (o nível geral de sincronização de cada pessoa), enquanto a diferença entre condições é menor e fica diluída. Ela existe dentro de cada pessoa (PC1 menor em EO no teste pareado, p < 0,001; ver `03_pca_node_features`), mas separa pouco as condições entre indivíduos.

### Regressão Logística com PCA e metadados

In [10]:
meta = df_dataset[META_FEATURES].copy()
meta["gender"] = pd.to_numeric(meta["gender"], errors="coerce")
meta = pd.get_dummies(meta, columns=["age"], drop_first=True)

X_meta = pd.concat([X, meta], axis=1)
y_meta = df_dataset.loc[X_meta.index, "y"]

pipe_pca_meta = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
    ("pca", PCA(n_components=10)),
    ("clf", LogisticRegression(max_iter=1000)),
])

evaluate_model(X_meta, y_meta, pipe_pca_meta)


=== Model Evaluation (GroupKFold por participante, 5 folds) ===
Mean Accuracy : 0.604 ± 0.065
Mean ROC-AUC  : 0.591 ± 0.109
Note: ROC-AUC values close to 0.5 indicate random performance.


{'accuracy_mean': np.float64(0.6036363636363635),
 'auc_mean': np.float64(0.5905454545454546)}

Acrescentar idade e gênero não melhorou o modelo (acurácia de 60,4% ± 6,5%; ROC-AUC de 0,591 ± 0,109). Isso era esperado: esses metadados são iguais no EO e no EC de cada pessoa e, portanto, não informam a condição.

Conclusão: as métricas de rede distinguem EO de EC entre participantes apenas de forma modesta (ROC-AUC ≈ 0,63 no melhor modelo). Por isso a classificação principal do trabalho é a de grupo etário (`07` → `08` → `09`), em que o sinal é mais forte.